# scProto (`ema`, soft) vs. SEACells niche recovery on `bk08` (s28nsc)

Tests whether scProto (BANKSY-lambda=0.8 affinity, kernel built with the real
SEACells package's own `SEACellGraph.rbf()`, `N_PROTOTYPES=200`) recovers
niche-transcriptional-program DGE signal (Branch 1/2), compared head-to-head
against SEACells trained on the same `bk08` graph and on the `arbf` reference
graph (SEACells baselines are trained in the separate `seacells_train.ipynb`
notebook; this notebook's eval cell reads their saved results directly).

Both `proto_usage_mode` variants (`ema`, `nk`) still train and cache their own
checkpoints below (cells `usage-mode-ema15`/`usage-mode-nkema15`), so neither
run needs to be redone later -- but only `ema` (soft-scored) is carried into
the scProto-vs-SEACells comparison from here on. `nk` did not reduce
prototype hub concentration relative to `ema`, and hard-assignment scoring is
excluded because it collapses to too few testable pairs to compare fairly
(`nk` hard: 1/146 pairs).

The DGE eval cell below reports two numbers: each run's own macro-average
over whatever pairs IT individually clears the Branch 1/2 gate on (shown for
transparency), and a matched-pairs version restricted to the intersection of
pairs every compared run clears -- matching `plan1_niche_recovery_eval.ipynb`'s
own "paired over the same N pairs" convention for head-to-head claims. Cite
the matched version, not the self-selected one.

Metacell-size-concentration numbers (`effective_n_metacells`, `top5_share`,
`gini`) are reported for context on every run -- useful background on how
each method spreads cells across its prototypes -- without a pass/fail
verdict attached.

`N_PROTOTYPES`=200 (not the dataset config default of 800) was chosen from an
earlier Leiden-community-count analysis of the graph, closest to the real
number of (cell type, niche) pairs in the ground truth.

Run this in Colab -- needs the Drive-mounted `data/` and `models/` folders.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Same base install as plan1_niche_recovery_eval.ipynb / fibroblast_scproto_vs_seacell.ipynb --
# no anndata version pin (see fibroblast_scproto_vs_seacell.ipynb's install-cell comment for
# why: no single anndata version satisfies both scarches/scvi-tools and SEACells -- the actual
# SEACells `dtype=` TypeError is patched at its call site in metacell_metrics.py instead).
!pip install -q scarches faiss-cpu scib-metrics
!pip install --upgrade --force-reinstall git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy statsmodels leidenalg python-igraph
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

**IMPORTANT: restart the runtime now** (Runtime -> Restart session) before running the cells below.

In [ ]:
_checks = {
    "numpy": "numpy", "scipy": "scipy", "anndata": "anndata", "scanpy": "scanpy",
    "scarches": "scarches", "scvi-tools": "scvi", "seacells": "SEACells",
    "palantir": "palantir", "scib-metrics": "scib_metrics",
    "statsmodels": "statsmodels", "faiss": "faiss", "leidenalg": "leidenalg",
}
_failed = []
for pkg, mod in _checks.items():
    try:
        __import__(mod)
    except ImportError as e:
        _failed.append((pkg, str(e)))and the
if _failed:
    print("FAILED imports (fix before continuing, then restart runtime again):")
    for pkg, err in _failed:
        print(f"  {pkg}: {err}")
else:
    print("All packages import cleanly.")

All packages import cleanly.


In [49]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [50]:
import os
import pickle

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import scipy.sparse as sp

from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import CODE_DIR, get_affinity_path, get_seacell_model_dir
from interpretable_ssl.augmenters.graph_generator import generate_affinity, _ensure_X_ctx
from interpretable_ssl.evaluation.batch_correct_baselines import run_seacells_on_latent
from interpretable_ssl.evaluation.niche_program_recovery import (
    compute_ground_truth, save_ground_truth, load_ground_truth,
    majority_label_metacells, load_cell_assignments,
    build_pseudobulk, branch1_recovery, branch2_recovery, macro_average,
    matched_macro_average, per_pair_diagnostics,
    load_soft_assignments, soft_label_metacells, build_soft_pseudobulk,
    size_concentration_summary, top_genes_for_pair,
    report_run, report_all, report_run_decoded, build_decoded_pseudobulk,
    coverage_penalized_average, coverage_penalized_detail,
)

DS_ID = 's28nsc'       # full (unsubsampled) slide 28 -- more cells per (cell type, niche)
                        # metacell group than ss28nsc.
CT_KEY = 'celltypes'
NICHE_KEY = 'niches_2D'
BATCH_KEY = DATASETS[DS_ID].get('batch_key')
K_GRAPH = 50            # matches configs/defaults.py's k_neighbors default
N_PROTOTYPES = DATASETS[DS_ID]['num_prototypes']

GT_PATH = os.path.join(CODE_DIR, 'files', 'celltype_niches_full_s28nsc.csv')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 500)
plt.rcParams['figure.dpi'] = 150

## Load dataset, persist calibrated X_ctx to disk (once), load ground truth

In [65]:
ds_conf = DATASETS[DS_ID]
adata = sc.read_h5ad(ds_conf['path'])
print(adata)
print('\ncell types:\n', adata.obs[CT_KEY].value_counts())
print('\nniches:\n', adata.obs[NICHE_KEY].value_counts(dropna=False))

AnnData object with n_obs Ã— n_vars = 58423 Ã— 960
    obs: 'section', 'celltypes', 'niches_3D', 'niches_2D', 'fibroblast_subclusters', 'tumor_pseudotime_rank', 'EMT_niche'
    uns: 'EMT_niche_colors', 'celltypes_colors', 'fibroblast_subclusters_colors', 'log1p', 'niches_2D_colors', 'niches_3D_colors', 'pca'
    obsm: 'X_covet', 'X_ctx', 'X_pca', 'X_umap_2D_neighbourhoods', 'X_umap_3D_neighbourhoods', 'X_umap_SCT', 'spatial'
    varm: 'PCs'
    layers: 'SCT', 'counts', None (.X)

cell types:
 celltypes
Fibroblasts                    15309
Tumor cells                    11382
Macrophages                     9390
Cytotoxic T cells               6011
Respiratory epithelium          2255
Vascular endothelium            2051
Monocytes                       1920
Plasma cells                    1637
Pericytes                       1483
Smooth muscle cells             1389
Regulatory T cells              1100
Cycling immune cells            1068
Dendritic cells                  864
Basal epith

In [66]:
# _ensure_X_ctx computes X_ctx (calibrated radius) only if not already present, then we
# write it back to the file on disk -- run_mc_task below reloads a FRESH copy of
# s28nsc.h5ad from disk for each separate scProto training call (bk32 and its
# saving here is what actually avoids recomputing X_ctx on every one of them. Same
# for the SEACells baseline calls further down, which reuse this notebook's in-memory
# `adata` directly (already has it after this cell either way).
s28nsc_path = ds_conf['path']
already_had_ctx = 'X_ctx' in adata.obsm

_ensure_X_ctx(adata)

if already_had_ctx:
    print('X_ctx was already present in the loaded adata -- assuming the file on disk '
          'already has it too, not re-saving.')
else:
    print(f'Writing X_ctx back to {s28nsc_path} so it is never recomputed again by this '
          f'or any future notebook run on s28nsc.')
    adata.write_h5ad(s28nsc_path)
    print('Saved.')

[X_ctx] using existing X_ctx already in adata (not recomputed)
X_ctx was already present in the loaded adata -- assuming the file on disk already has it too, not re-saving.


In [67]:
if os.path.exists(GT_PATH):
    print(f'Loading cached ground truth from {GT_PATH}')
    ground_truth = load_ground_truth(GT_PATH)
else:
    ground_truth = compute_ground_truth(adata, CT_KEY, NICHE_KEY, min_pos=5, min_ctrl=20)
    save_ground_truth(ground_truth, GT_PATH)
    n_rows = sum(len(v) for v in ground_truth.values())
    print(f'Saved full ground truth ({n_rows} gene rows across {len(ground_truth)} '
          f'(cell type, niche) pairs) to {GT_PATH}')

print(f'{len(ground_truth)} (cell type, niche) pairs pass the 5/20-cell minimum')

n_target = len(ground_truth)
print(f'Target: {n_target} real (cell type, niche) pairs in ground truth')


Loading cached ground truth from /content/drive/MyDrive/codes/interpretable-prototype/files/celltype_niches_full_s28nsc.csv
146 (cell type, niche) pairs pass the 5/20-cell minimum
Target: 146 real (cell type, niche) pairs in ground truth


## Fast path: `bk08`, N_PROTOTYPES=200 -- `ema` vs. `nk`, both at 15 epochs

Same graph (`bk08`) and `N_PROTOTYPES=200` throughout -- only `proto_usage_mode`
varies, both run the same 15 epochs for a fair, apples-to-apples comparison (no
epoch-count confound).

- **`ema`** (previous active default) -- max-based: satisfied by one confident
  cell per prototype, blind to total population.
- **`nk`** (now the default in `scproto.py`) -- sum-based aggregate mass,
  EMA-smoothed across batches so a genuinely rare population isn't penalized
  just for sampling unluckily into a low-count batch. Fixes `ema`'s blindness
  to total headcount.

Imports + training loop:

In [68]:
from interpretable_ssl.experiments.tasks import run_mc_task, LAMBDA_PROTO_UMAP_PRECON
from interpretable_ssl.evaluation.batch_correct_baselines import _topk_sparsify_rows, save_soft_assignments
import torch
import torch.nn.functional as F

trainers_usage_mode = {}


In [69]:
lc = dict(LAMBDA_PROTO_UMAP_PRECON)

exp_name = 'proto_bk08_np200_ema15'
try:
    print('Trying to load an existing ema/15ep checkpoint (no retraining)...')
    t, _, _ = run_mc_task(
        DS_ID, cvae_epochs=50, train_epochs=1, eval_freq=1, patience=1,
        batch_size=1024, affinity_type='bk08', lambda_config=lc,
        num_prototypes=200,
        trainer_kwargs={'experiment_name': exp_name},
        load_umap=True, skip_eval=True,
    )
    print('Loaded.')
except FileNotFoundError:
    print('=== training scProto: bk08, N_PROTOTYPES=200, proto_usage_mode=ema, 15 epochs ===')
    t, metrics, _ = run_mc_task(
        DS_ID, cvae_epochs=50, train_epochs=15, eval_freq=5, patience=10,
        batch_size=1024, lambda_config=lc, affinity_type='bk08',
        trainer_kwargs={'experiment_name': exp_name},
        label_key=CT_KEY, niche_key=NICHE_KEY, num_prototypes=200,
        skip_metrics=['task2', 'task3'],
    )
print(f'  epsilon={t.epsilon:.4f}')

model = t.model
model.eval()
with torch.no_grad():
    z_all = t.encode_adata(t.train_ds.adata, model, z_idx=1).to(t.device)
    soft = F.softmax(model.prototypes(z_all) / t.epsilon, dim=1)
soft_topk = _topk_sparsify_rows(soft.cpu().numpy(), k=20)
save_soft_assignments(t.get_dump_path(), soft_topk, t.train_ds.adata.obs_names.to_numpy())
del model, z_all, soft

trainers_usage_mode['proto_bk08_np200_ema15'] = t


Trying to load an existing ema/15ep checkpoint (no retraining)...
dataset is None, loading s28nsc
loading s28nsc data
âš ï¸ No HVG column found.
proto_bk08_np200_ema15_ds-s28n_NP200_prtInit-wayp_aff-bk08_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 

ðŸ“Š Affinity: wdeg[min/mean/max]=11.385/24.968/163.001, effk_med=59.0, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/s28nsc/pretrain/pretrain_ds-s28nsc_cvae_e50_NP200/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 's28nsc', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'num_prototypes': 200, 'condition_key': 'section'}
ðŸ“Š EdgeDataset: 3999862 edges
   Weigh

  0%|          | 0/58 [00:00<?, ?it/s]

  saved soft_assignments.npz (58423, 200) to /content/drive/MyDrive/models//s28nsc/proto_bk08_np200_ema15_ds-s28n_NP200_prtInit-wayp_aff-bk08_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31


In [70]:
lc = dict(LAMBDA_PROTO_UMAP_PRECON)
lc['proto_usage_mode'] = 'nk'  # nk = sum-based aggregate mass, EMA-smoothed (see scproto.py)

exp_name = 'proto_bk08_np200_nk15'
try:
    print('Trying to load an existing nk/15ep checkpoint (no retraining)...')
    t, _, _ = run_mc_task(
        DS_ID, cvae_epochs=50, train_epochs=1, eval_freq=1, patience=1,
        batch_size=1024, affinity_type='bk08', lambda_config=lc,
        num_prototypes=200,
        trainer_kwargs={'experiment_name': exp_name},
        load_umap=True, skip_eval=True,
    )
    print('Loaded.')
except FileNotFoundError:
    print('=== training scProto: bk08, N_PROTOTYPES=200, proto_usage_mode=nk, 15 epochs ===')
    t, metrics, _ = run_mc_task(
        DS_ID, cvae_epochs=50, train_epochs=15, eval_freq=5, patience=10,
        batch_size=1024, lambda_config=lc, affinity_type='bk08',
        trainer_kwargs={'experiment_name': exp_name},
        label_key=CT_KEY, niche_key=NICHE_KEY, num_prototypes=200,
        skip_metrics=['task2', 'task3'],
    )
print(f'  epsilon={t.epsilon:.4f}')

model = t.model
model.eval()
with torch.no_grad():
    z_all = t.encode_adata(t.train_ds.adata, model, z_idx=1).to(t.device)
    soft = F.softmax(model.prototypes(z_all) / t.epsilon, dim=1)
soft_topk = _topk_sparsify_rows(soft.cpu().numpy(), k=20)
save_soft_assignments(t.get_dump_path(), soft_topk, t.train_ds.adata.obs_names.to_numpy())
del model, z_all, soft

trainers_usage_mode['proto_bk08_np200_nk15'] = t


Trying to load an existing nk/15ep checkpoint (no retraining)...
dataset is None, loading s28nsc
loading s28nsc data
âš ï¸ No HVG column found.
proto_bk08_np200_nk15_ds-s28n_NP200_prtInit-wayp_aff-bk08_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 

ðŸ“Š Affinity: wdeg[min/mean/max]=11.385/24.968/163.001, effk_med=59.0, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/s28nsc/pretrain/pretrain_ds-s28nsc_cvae_e50_NP200/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 's28nsc', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'num_prototypes': 200, 'condition_key': 'section'}
ðŸ“Š EdgeDataset: 3999862 edges
   Weight range: [

  0%|          | 0/58 [00:00<?, ?it/s]

  saved soft_assignments.npz (58423, 200) to /content/drive/MyDrive/models//s28nsc/proto_bk08_np200_nk15_ds-s28n_NP200_prtInit-wayp_aff-bk08_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_lna1_nagg-max_upm-dotp_v31


## DGE eval: scProto (`ema`, soft) vs. SEACells (`arbf`/`bk08`, soft) -- matched pairs

Soft-assignment labeling only, and `nk` excluded from these tables (still
trained/cached above, just not shown here) -- for a clearer, less cluttered
comparison. SEACells uses its own archetypal soft-assignment matrix
(`model.A_`) the same way scProto uses its softmax, through the exact same
`soft_label_metacells`/`build_soft_pseudobulk` functions, so the comparison
is apples-to-apples by construction.

Two summary tables are printed: (1) each run's own macro-average over
whatever pairs IT individually clears the Branch 1/2 gate on -- reported for
transparency but NOT the fair comparison, since different runs can clear
different-sized, differently-easy subsets; (2) the matched-pairs version,
restricted to the intersection of pairs every compared run clears -- this is
the number to actually cite. Plus a detailed per-(cell type, niche) table:
number of positive metacells, their median/min size, and -- for context --
the REAL number of cells that actually exist for that (cell type, niche)
combination in the data (independent of any run), so a low score can be told
apart from "this pair just doesn't have much data to begin with."

In [71]:
run_dirs_eval = {name: t.get_dump_path() for name, t in trainers_usage_mode.items()
                 if 'nk' not in name}
run_dirs_eval['SEACells_arbf'] = get_seacell_model_dir(DS_ID, 'arbf')
run_dirs_eval['SEACells_bk08'] = get_seacell_model_dir(DS_ID, 'bk08')
missing_eval = {k: v for k, v in run_dirs_eval.items()
                if not v or not os.path.exists(os.path.join(v, 'cell_assignments.csv'))}
if missing_eval:
    print('WARNING -- missing cell_assignments.csv for:', missing_eval)

summary_eval, size_eval, per_pair_eval = report_all(run_dirs_eval, missing_eval, adata, ground_truth, CT_KEY, NICHE_KEY)

# Soft-only, ema-vs-SEACells comparison (nk and hard-mode excluded here for a
# clearer, less cluttered comparison -- nk is still trained/cached above, just
# not shown in these tables; hard mode is still computed by report_all
# internally, just filtered out of what's displayed/used below).
is_soft = summary_eval.index.str.contains(r'\(soft\)')
summary_eval = summary_eval[is_soft]
size_eval = size_eval[size_eval.index.str.contains(r'\(soft\)')]
per_pair_eval = per_pair_eval[per_pair_eval['run'].str.contains(r'\(soft\)')].reset_index(drop=True)

print()
print('Branch 1/2 -- each run scored on its OWN achievable pair set '
      '(reported for transparency, NOT the fair comparison -- see matched version below):')
display(summary_eval.round(3))

print()
print('Metacell size concentration, soft only (context, no verdict attached):')
display(size_eval.round(3))

# --- matched-pairs fair comparison ------------------------------------------------
# per_pair_eval has a row for every (cell_type, niche) with >=1 positive metacell, but
# pearson_r/kendall_tau/tpr are NaN wherever that run's branch1/branch2 gate wasn't
# cleared (min_pos_mc>=2, min_ctrl_mc>=2). Matching plan1_niche_recovery_eval.ipynb's
# own "paired over the same N pairs" precedent for head-to-head claims: restrict to
# the intersection of pairs EVERY compared run actually cleared, so no run's average
# is quietly computed over an easier or differently-sized subset than another's.
tested_pairs_by_run = {
    run: set(map(tuple, grp.dropna(subset=['pearson_r', 'kendall_tau', 'tpr'])[['cell_type', 'niche']].values))
    for run, grp in per_pair_eval.groupby('run')
}
matched_pairs = set.intersection(*tested_pairs_by_run.values()) if tested_pairs_by_run else set()
print()
print(f'Matched-pairs fair comparison: {len(matched_pairs)} (cell_type, niche) pairs '
      f'cleared the branch1/branch2 gate for EVERY run '
      f'({", ".join(f"{r}: {len(p)}" for r, p in tested_pairs_by_run.items())}).')


if matched_pairs:
    per_pair_matched = per_pair_eval[
        per_pair_eval.set_index(['cell_type', 'niche']).index.isin(matched_pairs)
    ].copy()
    summary_matched = matched_macro_average(per_pair_matched)
    summary_matched['n_pairs_tested'] = len(matched_pairs)
    print()
    print('Branch 1/2 -- MATCHED pairs only (the fair, head-to-head number -- cite this one):')
    display(summary_matched.round(3))
else:
    print('No pair clears the gate for every run -- no matched-pairs comparison possible.')

real_counts = (adata.obs.groupby([CT_KEY, NICHE_KEY]).size()
               .rename('n_real_cells').reset_index()
               .rename(columns={CT_KEY: 'cell_type', NICHE_KEY: 'niche'}))
per_pair_detail = per_pair_eval.merge(real_counts, on=['cell_type', 'niche'], how='left')
detail_cols = ['run', 'cell_type', 'niche', 'n_real_cells', 'n_pos_mc',
               'median_pos_mc_size', 'min_pos_mc_size', 'mean_niche_purity',
               'pearson_r', 'kendall_tau', 'tpr']

print()
print(f'Full per-(cell type, niche) x run detail (soft only) -- {len(per_pair_detail)} rows, '
      f'grouped by pair so every method sits together for the same pair:')
display(per_pair_detail[detail_cols].sort_values(['cell_type', 'niche', 'run']).round(3))

proto_bk08_np200_ema15: done
SEACells_arbf: done
SEACells_bk08: done

Branch 1/2 -- each run scored on its OWN achievable pair set (reported for transparency, NOT the fair comparison -- see matched version below):


,pearson_r,kendall_tau,tpr,n_pairs_tested
proto_bk08_np200_ema15 (soft),0.754,0.519,0.911,15.0
SEACells_arbf (soft),0.628,0.401,0.967,13.0
SEACells_bk08 (soft),0.728,0.463,0.969,16.0



Metacell size concentration, soft only (context, no verdict attached):


,n_metacells_used,median_size,gini,effective_n_metacells,top5_share_of_cells
run,,,,,
proto_bk08_np200_ema15 (soft),200,10.666,0.933,9.122,0.631
SEACells_arbf (soft),200,267.619,0.299,156.450,0.060
SEACells_bk08 (soft),200,250.320,0.292,155.391,0.066



Matched-pairs fair comparison: 7 (cell_type, niche) pairs cleared the branch1/branch2 gate for EVERY run (SEACells_arbf (soft): 13, SEACells_bk08 (soft): 16, proto_bk08_np200_ema15 (soft): 15).

Branch 1/2 -- MATCHED pairs only (the fair, head-to-head number -- cite this one):


,pearson_r,kendall_tau,tpr,n_pairs_tested
run,,,,
SEACells_arbf (soft),0.669,0.437,0.965,7
SEACells_bk08 (soft),0.714,0.453,0.970,7
proto_bk08_np200_ema15 (soft),0.677,0.455,0.912,7



Full per-(cell type, niche) x run detail (soft only) -- 91 rows, grouped by pair so every method sits together for the same pair:


,run,cell_type,niche,n_real_cells,n_pos_mc,median_pos_mc_size,min_pos_mc_size,mean_niche_purity,pearson_r,kendall_tau,tpr
59,SEACells_bk08 (soft),Alveolar cells,Alveolar spaces,444,1,135.875,135.875,0.844,NaN,NaN,NaN
0,proto_bk08_np200_ema15 (soft),Alveolar cells,Alveolar spaces,444,1,11.434,11.434,0.668,NaN,NaN,NaN
33,SEACells_arbf (soft),Basal epithelial cells,Airways,681,4,102.474,61.052,0.627,NaN,NaN,NaN
60,SEACells_bk08 (soft),Basal epithelial cells,Airways,681,4,125.347,111.882,0.846,NaN,NaN,NaN
1,proto_bk08_np200_ema15 (soft),Cytotoxic T cells,Airways,164,1,13.208,13.208,0.321,NaN,NaN,NaN
61,SEACells_bk08 (soft),Cytotoxic T cells,Alveolar spaces,225,1,91.910,91.910,0.704,NaN,NaN,NaN
2,proto_bk08_np200_ema15 (soft),Cytotoxic T cells,Alveolar spaces,225,1,9.561,9.561,0.208,NaN,NaN,NaN
62,SEACells_bk08 (soft),Cytotoxic T cells,Desmoplastic stroma,1702,2,252.307,225.251,0.415,0.718,0.482,1.000
3,proto_bk08_np200_ema15 (soft),Cytotoxic T cells,Desmoplastic stroma,1702,4,10.291,6.572,0.289,0.903,0.745,0.860
34,SEACells_arbf (soft),Cytotoxic T cells,Excluded,303,1,175.642,175.642,0.313,NaN,NaN,NaN


## Coverage-penalized average -- the number to actually cite

Self-selected and matched-pairs (both above) share a blind spot: a pair a run
can't produce a testable metacell for is simply DROPPED from that run's
average, not penalized -- neither number charges a method for failing to
cover a niche at all, only for accuracy on whatever it did attempt.

`coverage_penalized_average` fixes this: starts from all 146 real (cell type,
niche) pairs in `ground_truth`, scores 0 on pearson_r/kendall_tau/tpr for any
pair a run can't clear the Branch 1/2 gate for (instead of excluding it) --
but then drops any pair where EVERY compared run scores 0. A pair nobody can
test carries no information for comparing methods (everyone fails alike) and
would only dilute every average by the same fixed amount, so it isn't held
against any one method. A pair stays in as long as at least one run tested
it; every OTHER run that failed on that same pair still gets the 0 penalty.
`n_pairs_used`/`n_pairs_dropped_untestable_by_all` in the output show exactly
how many of the 146 survived this filter.

In [72]:
summary_coverage_penalized = coverage_penalized_average(per_pair_eval, ground_truth)
print("Branch 1/2 -- coverage-penalized (0 for any pair a run can't clear "
      "the gate for, pairs no run could test at all excluded) -- the fairest "
      "single number to cite:")
display(summary_coverage_penalized.round(3))

detail_coverage_penalized = coverage_penalized_detail(per_pair_eval, ground_truth)
print()
print('Per-(cell type, niche) detail, NOT aggregated -- every one of the 146 pairs, '
      'every run, 0.0 + tested=False where a run could not clear the gate:')
pivot_pearson = detail_coverage_penalized.pivot(index=['cell_type', 'niche'], columns='run', values='pearson_r')
display(pivot_pearson.round(3))

print()
print('Same grid, tested flag (True = real score behind the number, False = 0.0 penalty applied):')
pivot_tested = detail_coverage_penalized.pivot(index=['cell_type', 'niche'], columns='run', values='tested')
display(pivot_tested)

Branch 1/2 -- coverage-penalized (0 for any pair a run can't clear the gate for, pairs no run could test at all excluded) -- the fairest single number to cite:


,pearson_r,kendall_tau,tpr,n_pairs_tested,n_pairs_used,n_pairs_dropped_untestable_by_all
run,,,,,,
SEACells_arbf (soft),0.222,0.142,0.343,13,23,123
SEACells_bk08 (soft),0.493,0.318,0.648,16,23,123
proto_bk08_np200_ema15 (soft),0.506,0.348,0.596,15,23,123



Per-(cell type, niche) detail, NOT aggregated -- every one of the 146 pairs, every run, 0.0 + tested=False where a run could not clear the gate:


run                                                   SEACells_arbf (soft)  \
cell_type                   niche                                            
Alveolar cells              Airways                                  0.000   
                            Alveolar spaces                          0.000   
                            Desmoplastic stroma                      0.000   
                            Macrophage islands                       0.000   
                            T cell aggregates                        0.000   
                            Tumor surface                            0.000   
                            Vascular stroma                          0.000   
B cells                     Airways                                  0.000   
                            Alveolar spaces                          0.000   
                            Desmoplastic stroma                      0.000   
                            Macrophage islands                       0.000   
                            T cell aggregates                        0.000   
                            Tumor surface                            0.000   
                            Vascular stroma                          0.000   
Basal epithelial cells      Airways                                  0.000   
                            Desmoplastic stroma                      0.000   
                            Smooth muscle structures                 0.000   
                            T cell aggregates                        0.000   
                            Tumor surface                            0.000   
                            Vascular stroma                          0.000   
Cycling immune cells        Airways                                  0.000   
                            Alveolar spaces                          0.000   
                            Desmoplastic stroma                      0.000   
                            Macrophage islands                       0.000   
                            Smooth muscle structures                 0.000   
                            T cell aggregates                        0.000   
                            Tumor core                               0.000   
                            Tumor surface                            0.000   
                            Vascular stroma                          0.000   
Cytotoxic T cells           Airways                                  0.000   
                            Alveolar spaces                          0.000   
                            Desmoplastic stroma                      0.000   
                            Macrophage islands                       0.000   
                            Smooth muscle structures                 0.000   
                            T cell aggregates                        0.000   
                            Tumor core                               0.000   
                            Tumor surface                            0.000   
                            Vascular stroma                          0.000   
Dendritic cells             Airways                                  0.000   
                            Alveolar spaces                          0.000   
                            Desmoplastic stroma                      0.000   
                            Macrophage islands                       0.000   
                            T cell aggregates                        0.000   
                            Tumor core                               0.000   
                            Tumor surface                            0.000   
                            Vascular stroma                          0.000   
Fibroblasts                 Airways                                  0.782   
                            Alveolar spaces                          0.685   
                            Desmoplastic stroma                      0.799   
                      


Same grid, tested flag (True = real score behind the number, False = 0.0 penalty applied):


run                                                   SEACells_arbf (soft)  \
cell_type                   niche                                            
Alveolar cells              Airways                                  False   
                            Alveolar spaces                          False   
                            Desmoplastic stroma                      False   
                            Macrophage islands                       False   
                            T cell aggregates                        False   
                            Tumor surface                            False   
                            Vascular stroma                          False   
B cells                     Airways                                  False   
                            Alveolar spaces                          False   
                            Desmoplastic stroma                      False   
                            Macrophage islands                       False   
                            T cell aggregates                        False   
                            Tumor surface                            False   
                            Vascular stroma                          False   
Basal epithelial cells      Airways                                  False   
                            Desmoplastic stroma                      False   
                            Smooth muscle structures                 False   
                            T cell aggregates                        False   
                            Tumor surface                            False   
                            Vascular stroma                          False   
Cycling immune cells        Airways                                  False   
                            Alveolar spaces                          False   
                            Desmoplastic stroma                      False   
                            Macrophage islands                       False   
                            Smooth muscle structures                 False   
                            T cell aggregates                        False   
                            Tumor core                               False   
                            Tumor surface                            False   
                            Vascular stroma                          False   
Cytotoxic T cells           Airways                                  False   
                            Alveolar spaces                          False   
                            Desmoplastic stroma                      False   
                            Macrophage islands                       False   
                            Smooth muscle structures                 False   
                            T cell aggregates                        False   
                            Tumor core                               False   
                            Tumor surface                            False   
                            Vascular stroma                          False   
Dendritic cells             Airways                                  False   
                            Alveolar spaces                          False   
                            Desmoplastic stroma                      False   
                            Macrophage islands                       False   
                            T cell aggregates                        False   
                            Tumor core                               False   
                            Tumor surface                            False   
                            Vascular stroma                          False   
Fibroblasts                 Airways                                   True   
                            Alveolar spaces                           True   
                            Desmoplastic stroma                       True   
                      

## Coverage / union -- does scProto recover niches SEACells can't (or vice versa)?

Matched-pairs (above) answers "who's more accurate on shared ground" -- but by
construction it discards any pair only ONE method manages to produce a
testable positive+control group for (`min_pos_mc>=2`, `min_ctrl_mc>=2`).
That's a different, real axis: a method could look worse on matched-pairs
accuracy while still being valuable because it resolves niches the others
can't test at all.

This looks at the UNION of pairs tested by any run, reports each run's raw
coverage/recall (`n_pairs_tested / 146`), and -- the direct answer to "can
scProto find something SEACells can't" -- lists every pair tested by exactly
one run and nobody else, with that run's own score on it. Reuses
`tested_pairs_by_run`/`per_pair_eval`/`n_target` from the eval cell above (run
that cell first).

In [73]:
all_pairs_union = set.union(*tested_pairs_by_run.values())
print(f'Union: {len(all_pairs_union)}/{n_target} (cell_type, niche) pairs tested by AT LEAST ONE run.\n')

coverage_rows = []
for run, pairs in tested_pairs_by_run.items():
    others = set.union(*[p for r, p in tested_pairs_by_run.items() if r != run])
    unique = pairs - others
    coverage_rows.append({
        'run': run,
        'n_tested': len(pairs),
        'recall_of_146': len(pairs) / n_target,
        'n_unique_to_this_run': len(unique),
        'unique_pairs': sorted(unique),
    })
coverage_df = pd.DataFrame(coverage_rows).set_index('run')
print('Coverage / recall per run, and how many pairs ONLY that run tests:')
display(coverage_df[['n_tested', 'recall_of_146', 'n_unique_to_this_run']].round(3))

for _, row in coverage_df.iterrows():
    if row['n_unique_to_this_run'] == 0:
        continue
    print(f"\n{row.name} -- uniquely covers {row['n_unique_to_this_run']} pair(s) no other run tests:")
    for ct, niche in row['unique_pairs']:
        r = per_pair_eval[(per_pair_eval['run'] == row.name) &
                           (per_pair_eval['cell_type'] == ct) &
                           (per_pair_eval['niche'] == niche)]
        pr, tau, tpr_v = r['pearson_r'].values[0], r['kendall_tau'].values[0], r['tpr'].values[0]
        n_real = per_pair_detail[(per_pair_detail['cell_type'] == ct) &
                                  (per_pair_detail['niche'] == niche)]['n_real_cells'].values[0]
        print(f'  {ct} x {niche}: pearson_r={pr:.3f}  kendall_tau={tau:.3f}  tpr={tpr_v:.3f}  n_real_cells={n_real:.0f}')

Union: 23/146 (cell_type, niche) pairs tested by AT LEAST ONE run.

Coverage / recall per run, and how many pairs ONLY that run tests:


,n_tested,recall_of_146,n_unique_to_this_run
run,,,
SEACells_arbf (soft),13,0.089,1
SEACells_bk08 (soft),16,0.110,2
proto_bk08_np200_ema15 (soft),15,0.103,6



SEACells_arbf (soft) -- uniquely covers 1 pair(s) no other run tests:
  Tumor cells x Airways: pearson_r=0.098  kendall_tau=-0.024  tpr=0.965  n_real_cells=22

SEACells_bk08 (soft) -- uniquely covers 2 pair(s) no other run tests:
  Fibroblasts x Macrophage islands: pearson_r=0.714  kendall_tau=0.667  tpr=1.000  n_real_cells=1047
  Smooth muscle cells x Smooth muscle structures: pearson_r=0.767  kendall_tau=0.526  tpr=0.950  n_real_cells=733

proto_bk08_np200_ema15 (soft) -- uniquely covers 6 pair(s) no other run tests:
  Macrophages x Airways: pearson_r=0.459  kendall_tau=0.297  tpr=0.919  n_real_cells=310
  Macrophages x Alveolar spaces: pearson_r=0.916  kendall_tau=0.559  tpr=0.882  n_real_cells=388
  Macrophages x Desmoplastic stroma: pearson_r=0.797  kendall_tau=0.474  tpr=0.808  n_real_cells=1705
  Tumor cells x Desmoplastic stroma: pearson_r=0.783  kendall_tau=0.434  tpr=0.876  n_real_cells=865
  Vascular endothelium x T cell aggregates: pearson_r=0.942  kendall_tau=0.758  tpr=0

## Third mode: soft LABELS, but DECODED prototype expression (not real-cell pseudobulk)

`build_soft_pseudobulk` above uses soft assignment for BOTH labeling (which
niche/celltype a prototype represents) AND the expression profile itself (a
weighted sum of real cells' counts). This mode keeps soft assignment for
labeling only, and swaps in the model's own decoder output for each
prototype (`t.decode_prototypes()`) as its expression profile instead --
`report_run_decoded` (`niche_program_recovery.py`). This is a stronger test
of "scProto's prototypes encode niche-correlated transcriptional programs":
does the model's own generative reconstruction of a prototype contain the
right marker genes, independent of which real cells happened to carry soft
weight toward it. SEACells has no decoder, so there's no baseline-method
equivalent -- this section is scProto-only, reported alongside (not matched
against) the existing soft-real-pseudobulk numbers above.

In [74]:
t_ema = trainers_usage_mode['proto_bk08_np200_ema15']
summary_decoded, size_decoded, per_pair_decoded = report_run_decoded(
    'proto_bk08_np200_ema15', t_ema, adata, ground_truth, CT_KEY, NICHE_KEY,
)

print('Branch 1/2 -- soft-labels, decoded-profile:')
display(summary_decoded.round(3))

print()
print('Metacell size concentration -- soft-labels, decoded-profile:')
display(size_decoded.round(3))

print()
print('Same run, all three pseudobulk sources side by side (soft-real vs. decoded):')
compare_rows = pd.concat([summary_eval.copy(), summary_decoded])
display(compare_rows.round(3))

decoded_detail = per_pair_decoded.merge(real_counts, on=['cell_type', 'niche'], how='left')
decoded_candidates = decoded_detail[
    (decoded_detail['min_pos_mc_size'] >= 5) & (decoded_detail['n_real_cells'] >= 100)
].sort_values('pearson_r', ascending=False)
print()
print(f"{len(decoded_candidates)}/{len(decoded_detail)} decoded-profile pairs clear "
      f"the same filters as the candidates table above:")
display(decoded_candidates[detail_cols].head(15).round(3))

proto_bk08_np200_ema15: done (decoded-profile)
Branch 1/2 -- soft-labels, decoded-profile:


,pearson_r,kendall_tau,tpr,n_pairs_tested
"proto_bk08_np200_ema15 (soft-labels, decoded-profile)",0.384,0.279,0.897,15.0



Metacell size concentration -- soft-labels, decoded-profile:


,n_metacells_used,median_size,gini,effective_n_metacells,top5_share_of_cells
run,,,,,
"proto_bk08_np200_ema15 (soft-labels, decoded-profile)",200,10.666,0.933,9.122,0.631



Same run, all three pseudobulk sources side by side (soft-real vs. decoded):


,pearson_r,kendall_tau,tpr,n_pairs_tested
proto_bk08_np200_ema15 (soft),0.754,0.519,0.911,15.0
SEACells_arbf (soft),0.628,0.401,0.967,13.0
SEACells_bk08 (soft),0.728,0.463,0.969,16.0
"proto_bk08_np200_ema15 (soft-labels, decoded-profile)",0.384,0.279,0.897,15.0



29/33 decoded-profile pairs clear the same filters as the candidates table above:


,run,cell_type,niche,n_real_cells,n_pos_mc,median_pos_mc_size,min_pos_mc_size,mean_niche_purity,pearson_r,kendall_tau,tpr
12,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Fibroblasts,Tumor surface,3147,8,13.360,9.007,0.445,0.853,0.580,0.976
3,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Cytotoxic T cells,Desmoplastic stroma,1702,4,10.291,6.572,0.289,0.613,0.504,0.840
7,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Fibroblasts,Desmoplastic stroma,5856,19,15.108,6.350,0.496,0.458,0.314,0.989
13,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Fibroblasts,Vascular stroma,2179,3,21.454,15.842,0.269,0.428,0.308,1.000
31,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Vascular endothelium,T cell aggregates,183,2,15.096,13.699,0.380,0.406,0.287,0.824
18,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Macrophages,T cell aggregates,900,2,14.401,10.685,0.443,0.395,0.305,0.922
32,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Vascular endothelium,Vascular stroma,1079,24,10.580,6.208,0.603,0.350,0.295,0.802
25,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Tumor cells,Desmoplastic stroma,865,2,11.739,9.869,0.369,0.287,0.227,0.969
17,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Macrophages,Macrophage islands,2501,15,13.228,5.471,0.382,0.249,0.220,0.871
14,"proto_bk08_np200_ema15 (soft-labels, decoded-p...",Macrophages,Airways,310,2,8.348,7.181,0.368,0.248,0.200,0.865


## Where is scProto meaningfully better than SEACells (or vice versa)?

Restricted to non-degenerate, non-scarce pairs (`min_pos_mc_size>=5`,
`n_real_cells>=100`, same filter as the candidates table above), takes the
best `pearson_r` scProto achieves on each pair (across `ema`/`nk`, hard/soft)
vs. the best SEACells achieves (across `arbf`/`bk08`, hard/soft), and sorts by
the gap -- directly answers "is there a specific niche program scProto
actually recovers better than SEACells."

In [75]:
reliable = per_pair_detail[
    (per_pair_detail['min_pos_mc_size'] >= 5) & (per_pair_detail['n_real_cells'] >= 100)
].copy()

pivot = reliable.pivot_table(index=['cell_type', 'niche'], columns='run', values='pearson_r')
scproto_cols = [c for c in pivot.columns if c.startswith('proto_bk08')]
seacells_cols = [c for c in pivot.columns if c.startswith('SEACells')]

pivot['best_scProto'] = pivot[scproto_cols].max(axis=1)
pivot['best_SEACells'] = pivot[seacells_cols].max(axis=1)
pivot['scProto_advantage'] = pivot['best_scProto'] - pivot['best_SEACells']

print('Pairs where scProto beats SEACells by the largest margin:')
display(pivot.sort_values('scProto_advantage', ascending=False)
        [['best_scProto', 'best_SEACells', 'scProto_advantage']].head(15).round(3))

print()
print('Pairs where SEACells beats scProto by the largest margin:')
display(pivot.sort_values('scProto_advantage')
        [['best_scProto', 'best_SEACells', 'scProto_advantage']].head(15).round(3))


Pairs where scProto beats SEACells by the largest margin:


run                                         best_scProto  best_SEACells  \
cell_type         niche                                                   
Cytotoxic T cells Desmoplastic stroma              0.903          0.718   
Macrophages       Macrophage islands               0.764          0.654   
Fibroblasts       Desmoplastic stroma              0.879          0.809   
                  Tumor surface                    0.906          0.922   
Macrophages       T cell aggregates                0.817          0.877   
Cytotoxic T cells T cell aggregates                0.610          0.725   
Tumor cells       Tumor surface                    0.136          0.269   
Fibroblasts       Vascular stroma                  0.441          0.695   
                  T cell aggregates                  NaN          0.886   
                  Airways                            NaN          0.815   
                  Alveolar spaces                    NaN          0.882   
                  Macrophage islands                 NaN          0.714   
                  Smooth muscle structures           NaN          0.757   
Macrophages       Desmoplastic stroma              0.797            NaN   
                  Airways                          0.459            NaN   

run                                         scProto_advantage  
cell_type         niche                                        
Cytotoxic T cells Desmoplastic stroma                   0.185  
Macrophages       Macrophage islands                    0.109  
Fibroblasts       Desmoplastic stroma                   0.070  
                  Tumor surface                        -0.016  
Macrophages       T cell aggregates                    -0.059  
Cytotoxic T cells T cell aggregates                    -0.115  
Tumor cells       Tumor surface                        -0.132  
Fibroblasts       Vascular stroma                      -0.254  
                  T cell aggregates                       NaN  
                  Airways                                 NaN  
                  Alveolar spaces                         NaN  
                  Macrophage islands                      NaN  
                  Smooth muscle structures                NaN  
Macrophages       Desmoplastic stroma                     NaN  
                  Airways                                 NaN


Pairs where SEACells beats scProto by the largest margin:


run                                         best_scProto  best_SEACells  \
cell_type         niche                                                   
Fibroblasts       Vascular stroma                  0.441          0.695   
Tumor cells       Tumor surface                    0.136          0.269   
Cytotoxic T cells T cell aggregates                0.610          0.725   
Macrophages       T cell aggregates                0.817          0.877   
Fibroblasts       Tumor surface                    0.906          0.922   
                  Desmoplastic stroma              0.879          0.809   
Macrophages       Macrophage islands               0.764          0.654   
Cytotoxic T cells Desmoplastic stroma              0.903          0.718   
Fibroblasts       T cell aggregates                  NaN          0.886   
                  Airways                            NaN          0.815   
                  Alveolar spaces                    NaN          0.882   
                  Macrophage islands                 NaN          0.714   
                  Smooth muscle structures           NaN          0.757   
Macrophages       Desmoplastic stroma              0.797            NaN   
                  Airways                          0.459            NaN   

run                                         scProto_advantage  
cell_type         niche                                        
Fibroblasts       Vascular stroma                      -0.254  
Tumor cells       Tumor surface                        -0.132  
Cytotoxic T cells T cell aggregates                    -0.115  
Macrophages       T cell aggregates                    -0.059  
Fibroblasts       Tumor surface                        -0.016  
                  Desmoplastic stroma                   0.070  
Macrophages       Macrophage islands                    0.109  
Cytotoxic T cells Desmoplastic stroma                   0.185  
Fibroblasts       T cell aggregates                       NaN  
                  Airways                                 NaN  
                  Alveolar spaces                         NaN  
                  Macrophage islands                      NaN  
                  Smooth muscle structures                NaN  
Macrophages       Desmoplastic stroma                     NaN  
                  Airways                                 NaN

## Good (cell type, niche) candidates for literature validation

Filters the ema/nk per-pair table down to pairs that are (a) not singleton-
driven (`min_pos_mc_size >= 5`) and (b) not just data-scarcity artifacts
(`n_real_cells >= 100`), then sorts by `pearson_r` descending -- these are the
pairs where the model's metacell-level logFC most closely tracks the real
single-cell ground truth, on real, non-degenerate metacells. Good starting
points to check the actual genes against the literature.

In [76]:
real_counts = (adata.obs.groupby([CT_KEY, NICHE_KEY]).size()
               .rename('n_real_cells').reset_index()
               .rename(columns={CT_KEY: 'cell_type', NICHE_KEY: 'niche'}))
per_pair_eval_detail = per_pair_eval.merge(real_counts, on=['cell_type', 'niche'], how='left')
detail_cols = ['run', 'cell_type', 'niche', 'n_real_cells', 'n_pos_mc',
               'median_pos_mc_size', 'min_pos_mc_size', 'mean_niche_purity',
               'pearson_r', 'kendall_tau', 'tpr']

candidates = per_pair_eval_detail[
    (per_pair_eval_detail['min_pos_mc_size'] >= 5) &
    (per_pair_eval_detail['n_real_cells'] >= 100)
].sort_values('pearson_r', ascending=False)

print(f'{len(candidates)}/{len(per_pair_eval_detail)} pairs clear both filters '
      f'(min_pos_mc_size>=5, n_real_cells>=100)')
candidates[detail_cols].head(15).round(3)


85/91 pairs clear both filters (min_pos_mc_size>=5, n_real_cells>=100)


,run,cell_type,niche,n_real_cells,n_pos_mc,median_pos_mc_size,min_pos_mc_size,mean_niche_purity,pearson_r,kendall_tau,tpr
31,proto_bk08_np200_ema15 (soft),Vascular endothelium,T cell aggregates,183,2,15.096,13.699,0.380,0.942,0.758,0.971
73,SEACells_bk08 (soft),Fibroblasts,Tumor surface,3147,9,432.671,141.480,0.554,0.922,0.687,0.976
55,SEACells_arbf (soft),Tumor cells,Tumor core,3596,9,424.638,243.887,0.465,0.919,0.523,0.976
15,proto_bk08_np200_ema15 (soft),Macrophages,Alveolar spaces,388,3,10.942,6.897,0.293,0.916,0.559,0.882
32,proto_bk08_np200_ema15 (soft),Vascular endothelium,Vascular stroma,1079,24,10.580,6.208,0.603,0.915,0.651,0.958
12,proto_bk08_np200_ema15 (soft),Fibroblasts,Tumor surface,3147,8,13.360,9.007,0.445,0.906,0.665,0.976
87,SEACells_bk08 (soft),Tumor cells,Tumor core,3596,12,342.695,95.116,0.578,0.904,0.518,0.966
3,proto_bk08_np200_ema15 (soft),Cytotoxic T cells,Desmoplastic stroma,1702,4,10.291,6.572,0.289,0.903,0.745,0.860
41,SEACells_arbf (soft),Fibroblasts,T cell aggregates,1001,8,431.906,160.442,0.305,0.886,0.604,0.941
67,SEACells_bk08 (soft),Fibroblasts,Alveolar spaces,415,9,187.489,114.051,0.535,0.882,0.646,0.974


## Top genes behind the best candidates -- copy these into a literature search

For the top few candidate (run, cell_type, niche) rows above, pulls the actual
per-gene logFC comparison (metacell-level vs. real single-cell-level, restricted
to genes already significant in the ground truth) and shows the genes with the
largest metacell-level effect, flagging whether the direction (up/down) agrees
with the real single-cell result. Rebuilds each run's pseudobulk/labels once
from its saved `cell_assignments.csv` -- no retraining, no re-encoding.

In [77]:
N_TOP_PAIRS = 5

for _, row in candidates.head(N_TOP_PAIRS).iterrows():
    run_name = row['run'].split(' (')[0]
    hard_or_soft = 'soft' if '(soft)' in row['run'] else 'hard'
    run_dir = run_dirs_eval[run_name]

    cell_assign = load_cell_assignments(run_dir, adata, NICHE_KEY)
    if hard_or_soft == 'hard':
        mc_labels = majority_label_metacells(cell_assign, CT_KEY, NICHE_KEY)
        pseudobulk = build_pseudobulk(cell_assign, adata)
    else:
        S, cell_ids = load_soft_assignments(run_dir)
        mc_labels = soft_label_metacells(S, cell_ids, adata, CT_KEY, NICHE_KEY)
        pseudobulk = build_soft_pseudobulk(S, cell_ids, adata)

    genes = top_genes_for_pair(pseudobulk, mc_labels, ground_truth,
                                row['cell_type'], row['niche'], top_n=15)
    print(f"\n=== {row['run']} -- {row['cell_type']} x {row['niche']} "
          f"(pearson_r={row['pearson_r']:.3f}, n_real_cells={row['n_real_cells']:.0f}) ===")
    display(genes.round(3))



=== proto_bk08_np200_ema15 (soft) -- Vascular endothelium x T cell aggregates (pearson_r=0.942, n_real_cells=183) ===


,gene,mc_logfc,sc_logfc,same_direction
1,ACKR1,3.569,2.748,True
7,KDR,-2.338,-1.831,True
2,CLU,2.128,2.472,True
5,HLA.DRA,1.881,1.912,True
0,CD74,1.842,3.004,True
4,FLT1,-1.781,-1.981,True
11,SPRY4,-1.769,-1.653,True
23,PDGFB,-1.756,-1.308,True
15,IL1R1,1.733,1.543,True
30,BGN,-1.729,-0.940,True



=== SEACells_bk08 (soft) -- Fibroblasts x Tumor surface (pearson_r=0.922, n_real_cells=3147) ===


,gene,mc_logfc,sc_logfc,same_direction
12,IGHA1,-1.915,-1.226,True
5,IGKC,-1.839,-1.503,True
9,IGHG2,-1.728,-1.310,True
7,IGHG1,-1.708,-1.380,True
1,PTGDS,-1.452,-2.160,True
0,IGFBP5,1.366,2.252,True
11,CLU,-1.266,-1.274,True
4,COL11A1,1.172,1.519,True
14,CFD,-1.027,-1.141,True
10,SELENOP,-1.016,-1.304,True



=== SEACells_arbf (soft) -- Tumor cells x Tumor core (pearson_r=0.919, n_real_cells=3596) ===


,gene,mc_logfc,sc_logfc,same_direction
18,IGFBP7,-0.804,-0.773,True
52,LYZ,-0.731,-0.591,True
30,C1QB,-0.605,-0.650,True
0,LUM,-0.576,-1.301,True
16,COL1A2,-0.574,-0.832,True
10,MMP2,-0.541,-0.882,True
63,CD68,-0.539,-0.569,True
5,COL3A1,-0.531,-0.979,True
79,C1QA,-0.519,-0.518,True
17,IGKC,-0.517,-0.815,True



=== proto_bk08_np200_ema15 (soft) -- Macrophages x Alveolar spaces (pearson_r=0.916, n_real_cells=388) ===


,gene,mc_logfc,sc_logfc,same_direction
0,SPP1,-2.562,-3.695,True
4,MT2A,-1.301,-0.948,True
8,LGALS1,-1.292,-0.727,True
2,COL1A1,-1.211,-1.379,True
9,GLUL,-0.887,-0.622,True
5,JUNB,-0.768,-0.834,True
7,GSN,-0.633,-0.817,True
6,FOS,-0.573,0.823,False
1,CCL18,0.471,1.476,True
13,C1QA,-0.215,0.433,False



=== proto_bk08_np200_ema15 (soft) -- Vascular endothelium x Vascular stroma (pearson_r=0.915, n_real_cells=1079) ===


,gene,mc_logfc,sc_logfc,same_direction
9,ACKR1,-3.339,-1.546,True
8,CLU,-2.529,-1.577,True
1,KDR,2.306,2.268,True
6,CXCR4,2.112,1.735,True
2,FLT1,2.050,2.219,True
12,ACE,2.020,1.471,True
5,SPRY4,2.008,1.755,True
26,PTGDS,-1.996,-1.170,True
55,IL1R1,-1.918,-0.843,True
0,MMP9,1.896,2.542,True


## Same evaluation, against `niches_3D` ground truth instead of `niches_2D`

Everything above stays as-is (2D results untouched). This repeats the exact
same soft-only, ema-vs-SEACells comparison, but scoring the SAME already-
trained/already-run metacells (`run_dirs_eval` -- no retraining, no rerunning
SEACells, graphs/checkpoints don't depend on which niche label they're scored
against) against `niches_3D` instead.

In [78]:
NICHE_KEY_3D = 'niches_3D'
GT_PATH_3D = os.path.join(CODE_DIR, 'files', 'celltype_niches_3d_full_s28nsc.csv')

if os.path.exists(GT_PATH_3D):
    print(f'Loading cached 3D ground truth from {GT_PATH_3D}')
    ground_truth_3d = load_ground_truth(GT_PATH_3D)
else:
    ground_truth_3d = compute_ground_truth(adata, CT_KEY, NICHE_KEY_3D, min_pos=5, min_ctrl=20)
    save_ground_truth(ground_truth_3d, GT_PATH_3D)

n_target_3d = len(ground_truth_3d)
print(f'Target (3D niches): {n_target_3d} real (cell type, niche) pairs in ground truth')


Loading cached 3D ground truth from /content/drive/MyDrive/codes/interpretable-prototype/files/celltype_niches_3d_full_s28nsc.csv
Target (3D niches): 150 real (cell type, niche) pairs in ground truth


In [79]:
summary_eval_3d, size_eval_3d, per_pair_eval_3d = report_all(
    run_dirs_eval, missing_eval, adata, ground_truth_3d, ct_key=CT_KEY, niche_key=NICHE_KEY_3D)

summary_eval_3d = summary_eval_3d[summary_eval_3d.index.str.contains(r'\(soft\)')]
size_eval_3d = size_eval_3d[size_eval_3d.index.str.contains(r'\(soft\)')]
per_pair_eval_3d = per_pair_eval_3d[per_pair_eval_3d['run'].str.contains(r'\(soft\)')].reset_index(drop=True)

print()
print('Branch 1/2, soft only (ema vs. SEACells) -- niches_3D:')
display(summary_eval_3d.round(3))

print()
print('Metacell size concentration, soft only -- niches_3D:')
display(size_eval_3d.round(3))

real_counts_3d = (adata.obs.groupby([CT_KEY, NICHE_KEY_3D]).size()
                  .rename('n_real_cells').reset_index()
                  .rename(columns={CT_KEY: 'cell_type', NICHE_KEY_3D: 'niche'}))
per_pair_detail_3d = per_pair_eval_3d.merge(real_counts_3d, on=['cell_type', 'niche'], how='left')

print()
print(f'Full per-(cell type, niche) x run detail (soft only, niches_3D) -- {len(per_pair_detail_3d)} rows:')
display(per_pair_detail_3d[detail_cols].sort_values(['cell_type', 'niche', 'run']).round(3))


proto_bk08_np200_ema15: done
SEACells_arbf: done
SEACells_bk08: done

Branch 1/2, soft only (ema vs. SEACells) -- niches_3D:


,pearson_r,kendall_tau,tpr,n_pairs_tested
proto_bk08_np200_ema15 (soft),0.651,0.434,0.903,19.0
SEACells_arbf (soft),0.673,0.442,0.949,18.0
SEACells_bk08 (soft),0.657,0.433,0.944,19.0



Metacell size concentration, soft only -- niches_3D:


,n_metacells_used,median_size,gini,effective_n_metacells,top5_share_of_cells
run,,,,,
proto_bk08_np200_ema15 (soft),200,10.666,0.933,9.122,0.631
SEACells_arbf (soft),200,267.619,0.299,156.450,0.060
SEACells_bk08 (soft),200,250.320,0.292,155.391,0.066



Full per-(cell type, niche) x run detail (soft only, niches_3D) -- 94 rows:


,run,cell_type,niche,n_real_cells,n_pos_mc,median_pos_mc_size,min_pos_mc_size,mean_niche_purity,pearson_r,kendall_tau,tpr
61,SEACells_bk08 (soft),Alveolar cells,Alveolar spaces,438,1,135.875,135.875,0.649,NaN,NaN,NaN
0,proto_bk08_np200_ema15 (soft),Alveolar cells,Alveolar spaces,438,1,11.434,11.434,0.641,NaN,NaN,NaN
31,SEACells_arbf (soft),Basal epithelial cells,Airways,691,4,102.474,61.052,0.637,NaN,NaN,NaN
62,SEACells_bk08 (soft),Basal epithelial cells,Airways,691,4,125.347,111.882,0.857,NaN,NaN,NaN
1,proto_bk08_np200_ema15 (soft),Cytotoxic T cells,Airways,421,4,10.988,9.561,0.259,0.701,0.278,0.889
2,proto_bk08_np200_ema15 (soft),Cytotoxic T cells,Desmoplastic stroma,1153,2,8.521,6.572,0.400,0.752,0.626,0.818
32,SEACells_arbf (soft),Cytotoxic T cells,Excluded,303,1,175.642,175.642,0.313,NaN,NaN,NaN
63,SEACells_bk08 (soft),Cytotoxic T cells,Smooth muscle structures,210,2,238.860,234.707,0.351,0.779,0.571,1.000
33,SEACells_arbf (soft),Cytotoxic T cells,T cell aggregates,2449,10,339.294,197.880,0.618,NaN,NaN,NaN
64,SEACells_bk08 (soft),Cytotoxic T cells,T cell aggregates,2449,12,343.462,91.910,0.643,0.150,0.092,0.837


## Where is scProto meaningfully better than SEACells -- `niches_3D`

In [80]:
reliable_3d = per_pair_detail_3d[
    (per_pair_detail_3d['min_pos_mc_size'] >= 5) & (per_pair_detail_3d['n_real_cells'] >= 100)
].copy()

pivot_3d = reliable_3d.pivot_table(index=['cell_type', 'niche'], columns='run', values='pearson_r')
scproto_cols_3d = [c for c in pivot_3d.columns if c.startswith('proto_bk08')]
seacells_cols_3d = [c for c in pivot_3d.columns if c.startswith('SEACells')]

pivot_3d['best_scProto'] = pivot_3d[scproto_cols_3d].max(axis=1)
pivot_3d['best_SEACells'] = pivot_3d[seacells_cols_3d].max(axis=1)
pivot_3d['scProto_advantage'] = pivot_3d['best_scProto'] - pivot_3d['best_SEACells']

print('Pairs where scProto beats SEACells by the largest margin (niches_3D):')
display(pivot_3d.sort_values('scProto_advantage', ascending=False)
        [['best_scProto', 'best_SEACells', 'scProto_advantage']].head(15).round(3))

print()
print('Pairs where SEACells beats scProto by the largest margin (niches_3D):')
display(pivot_3d.sort_values('scProto_advantage')
        [['best_scProto', 'best_SEACells', 'scProto_advantage']].head(15).round(3))


Pairs where scProto beats SEACells by the largest margin (niches_3D):


run                                           best_scProto  best_SEACells  \
cell_type           niche                                                   
Cytotoxic T cells   T cell aggregates                0.600          0.150   
Fibroblasts         Airways                          0.862          0.760   
Smooth muscle cells Airways                          0.946          0.879   
Macrophages         Alveolar spaces                  0.798          0.805   
                    T cell aggregates                0.847          0.918   
Fibroblasts         Desmoplastic stroma              0.724          0.842   
Macrophages         Desmoplastic stroma              0.599          0.817   
                    Tumor surface                    0.689          0.914   
Fibroblasts         Alveolar spaces                  0.623          0.865   
Tumor cells         Tumor surface                   -0.121          0.230   
Smooth muscle cells Smooth muscle structures        -0.127          0.596   
Cytotoxic T cells   Airways                          0.701            NaN   
                    Desmoplastic stroma              0.752            NaN   
                    Smooth muscle structures           NaN          0.779   
Fibroblasts         Smooth muscle structures           NaN          0.867   

run                                           scProto_advantage  
cell_type           niche                                        
Cytotoxic T cells   T cell aggregates                     0.450  
Fibroblasts         Airways                               0.101  
Smooth muscle cells Airways                               0.067  
Macrophages         Alveolar spaces                      -0.007  
                    T cell aggregates                    -0.071  
Fibroblasts         Desmoplastic stroma                  -0.118  
Macrophages         Desmoplastic stroma                  -0.218  
                    Tumor surface                        -0.224  
Fibroblasts         Alveolar spaces                      -0.242  
Tumor cells         Tumor surface                        -0.352  
Smooth muscle cells Smooth muscle structures             -0.723  
Cytotoxic T cells   Airways                                 NaN  
                    Desmoplastic stroma                     NaN  
                    Smooth muscle structures                NaN  
Fibroblasts         Smooth muscle structures                NaN


Pairs where SEACells beats scProto by the largest margin (niches_3D):


run                                           best_scProto  best_SEACells  \
cell_type           niche                                                   
Smooth muscle cells Smooth muscle structures        -0.127          0.596   
Tumor cells         Tumor surface                   -0.121          0.230   
Fibroblasts         Alveolar spaces                  0.623          0.865   
Macrophages         Tumor surface                    0.689          0.914   
                    Desmoplastic stroma              0.599          0.817   
Fibroblasts         Desmoplastic stroma              0.724          0.842   
Macrophages         T cell aggregates                0.847          0.918   
                    Alveolar spaces                  0.798          0.805   
Smooth muscle cells Airways                          0.946          0.879   
Fibroblasts         Airways                          0.862          0.760   
Cytotoxic T cells   T cell aggregates                0.600          0.150   
                    Airways                          0.701            NaN   
                    Desmoplastic stroma              0.752            NaN   
                    Smooth muscle structures           NaN          0.779   
Fibroblasts         Smooth muscle structures           NaN          0.867   

run                                           scProto_advantage  
cell_type           niche                                        
Smooth muscle cells Smooth muscle structures             -0.723  
Tumor cells         Tumor surface                        -0.352  
Fibroblasts         Alveolar spaces                      -0.242  
Macrophages         Tumor surface                        -0.224  
                    Desmoplastic stroma                  -0.218  
Fibroblasts         Desmoplastic stroma                  -0.118  
Macrophages         T cell aggregates                    -0.071  
                    Alveolar spaces                      -0.007  
Smooth muscle cells Airways                               0.067  
Fibroblasts         Airways                               0.101  
Cytotoxic T cells   T cell aggregates                     0.450  
                    Airways                                 NaN  
                    Desmoplastic stroma                     NaN  
                    Smooth muscle structures                NaN  
Fibroblasts         Smooth muscle structures                NaN

## Is the collapse solved? -- `niches_3D`

In [81]:
print(f'N_PROTOTYPES=200, n_target_3d (real ct-niche_3D pairs)={n_target_3d}')
print()

verdict_rows_3d = []
for run in size_eval_3d.index:
    eff_n = size_eval_3d.loc[run, 'effective_n_metacells']
    top5 = size_eval_3d.loc[run, 'top5_share_of_cells']
    gini = size_eval_3d.loc[run, 'gini']
    if eff_n >= 0.8 * n_target_3d:
        verdict = 'SOLVED (effective_n clears n_target)'
    elif eff_n >= 0.4 * n_target_3d:
        verdict = 'PARTIAL (effective_n is meaningfully below n_target)'
    else:
        verdict = 'STILL COLLAPSED (effective_n far below n_target)'
    verdict_rows_3d.append({'run': run, 'effective_n': round(eff_n, 1),
                             'n_target': n_target_3d, 'top5_share': round(top5, 3),
                             'gini': round(gini, 3), 'verdict': verdict})
    print(f'{run:35s} effective_n={eff_n:6.1f}  top5_share={top5:.1%}  gini={gini:.3f}  -> {verdict}')

pd.DataFrame(verdict_rows_3d).set_index('run')


N_PROTOTYPES=200, n_target_3d (real ct-niche_3D pairs)=150

proto_bk08_np200_ema15 (soft)       effective_n=   9.1  top5_share=63.1%  gini=0.933  -> STILL COLLAPSED (effective_n far below n_target)
SEACells_arbf (soft)                effective_n= 156.5  top5_share=6.0%  gini=0.299  -> SOLVED (effective_n clears n_target)
SEACells_bk08 (soft)                effective_n= 155.4  top5_share=6.6%  gini=0.292  -> SOLVED (effective_n clears n_target)


,effective_n,n_target,top5_share,gini,verdict
run,,,,,
proto_bk08_np200_ema15 (soft),9.1,150,0.631,0.933,STILL COLLAPSED (effective_n far below n_target)
SEACells_arbf (soft),156.5,150,0.060,0.299,SOLVED (effective_n clears n_target)
SEACells_bk08 (soft),155.4,150,0.066,0.292,SOLVED (effective_n clears n_target)


## Good (cell type, niche) candidates for literature validation -- `niches_3D`

In [82]:
candidates_3d = per_pair_detail_3d[
    (per_pair_detail_3d['min_pos_mc_size'] >= 5) &
    (per_pair_detail_3d['n_real_cells'] >= 100)
].sort_values('pearson_r', ascending=False)

print(f'{len(candidates_3d)}/{len(per_pair_detail_3d)} pairs clear both filters '
      f'(min_pos_mc_size>=5, n_real_cells>=100)')
candidates_3d[detail_cols].head(15).round(3)


90/94 pairs clear both filters (min_pos_mc_size>=5, n_real_cells>=100)


,run,cell_type,niche,n_real_cells,n_pos_mc,median_pos_mc_size,min_pos_mc_size,mean_niche_purity,pearson_r,kendall_tau,tpr
21,proto_bk08_np200_ema15 (soft),Smooth muscle cells,Airways,220,6,12.634,9.751,0.598,0.946,0.628,0.957
27,proto_bk08_np200_ema15 (soft),Vascular endothelium,Airways,151,2,16.990,15.708,0.413,0.932,0.572,0.958
30,proto_bk08_np200_ema15 (soft),Vascular endothelium,Vascular stroma,1007,22,10.201,6.208,0.623,0.919,0.663,0.931
79,SEACells_bk08 (soft),Macrophages,T cell aggregates,1579,4,257.801,153.193,0.518,0.918,0.647,0.983
81,SEACells_bk08 (soft),Macrophages,Tumor surface,1950,8,257.141,180.384,0.548,0.914,0.445,0.946
48,SEACells_arbf (soft),Macrophages,T cell aggregates,1579,3,284.333,214.893,0.615,0.913,0.663,0.991
29,proto_bk08_np200_ema15 (soft),Vascular endothelium,T cell aggregates,232,3,13.699,13.050,0.348,0.913,0.749,0.975
49,SEACells_arbf (soft),Macrophages,Tumor surface,1950,7,288.372,207.881,0.447,0.900,0.356,0.950
84,SEACells_bk08 (soft),Smooth muscle cells,Airways,220,3,92.584,84.281,0.428,0.879,0.621,1.000
72,SEACells_bk08 (soft),Fibroblasts,Tumor core,1015,3,423.516,300.116,0.443,0.871,0.610,0.991


## Top genes behind the best `niches_3D` candidates -- copy these into a literature search

In [83]:
N_TOP_PAIRS = 5

for _, row in candidates_3d.head(N_TOP_PAIRS).iterrows():
    run_name = row['run'].split(' (')[0]
    run_dir = run_dirs_eval[run_name]

    S, cell_ids = load_soft_assignments(run_dir)
    mc_labels = soft_label_metacells(S, cell_ids, adata, CT_KEY, NICHE_KEY_3D)
    pseudobulk = build_soft_pseudobulk(S, cell_ids, adata)

    genes = top_genes_for_pair(pseudobulk, mc_labels, ground_truth_3d,
                                row['cell_type'], row['niche'], top_n=15)
    print()
    print(f"run: {row['run']} -- {row['cell_type']} x {row['niche']} "
          f"(pearson_r={row['pearson_r']:.3f}, n_real_cells={row['n_real_cells']:.0f})")
    display(genes.round(3))



run: proto_bk08_np200_ema15 (soft) -- Smooth muscle cells x Airways (pearson_r=0.946, n_real_cells=220)


,gene,mc_logfc,sc_logfc,same_direction
4,DUSP1,-1.759,-2.874,True
2,ACTG2,1.758,2.956,True
10,MYL9,1.507,2.234,True
12,ZFP36,-1.398,-1.959,True
17,JUNB,-1.370,-1.757,True
7,TPM1,1.295,2.429,True
6,TPM2,1.245,2.463,True
1,MYH11,1.214,2.974,True
9,ADIRF,-1.117,-2.350,True
15,COL3A1,-1.082,-1.882,True



run: proto_bk08_np200_ema15 (soft) -- Vascular endothelium x Airways (pearson_r=0.932, n_real_cells=151)


,gene,mc_logfc,sc_logfc,same_direction
10,IGFBP3,-2.176,-1.588,True
7,FLT1,-2.171,-1.708,True
5,KDR,-2.100,-1.809,True
15,SPRY4,-2.094,-1.444,True
2,CLU,2.027,2.252,True
0,PTGDS,1.947,2.770,True
16,COL3A1,-1.788,-1.329,True
4,ACE,-1.691,-2.070,True
17,INSR,-1.632,-1.315,True
6,PDGFB,-1.579,-1.805,True



run: proto_bk08_np200_ema15 (soft) -- Vascular endothelium x Vascular stroma (pearson_r=0.919, n_real_cells=1007)


,gene,mc_logfc,sc_logfc,same_direction
5,ACKR1,-3.133,-2.320,True
2,CLU,-2.432,-2.540,True
8,PTGDS,-2.268,-2.033,True
27,IL1R1,-2.174,-1.376,True
7,CXCR4,2.050,2.174,True
1,KDR,2.016,2.597,True
38,TNFRSF4,1.970,1.234,True
0,FLT1,1.959,2.686,True
17,ANGPT2,1.916,1.673,True
4,IGFBP3,1.760,2.349,True



run: SEACells_bk08 (soft) -- Macrophages x T cell aggregates (pearson_r=0.918, n_real_cells=1579)


,gene,mc_logfc,sc_logfc,same_direction
0,SPP1,-2.134,-2.750,True
1,CCL19,1.814,2.017,True
7,IGKC,1.462,1.223,True
17,CCL5,1.283,1.038,True
3,CXCL10,1.217,1.811,True
2,CXCL9,1.194,1.951,True
11,PTGDS,1.157,1.090,True
15,IDO1,0.994,1.071,True
31,IGHG2,0.964,0.878,True
52,ITGAM,-0.852,-0.659,True



run: SEACells_bk08 (soft) -- Macrophages x Tumor surface (pearson_r=0.914, n_real_cells=1950)


,gene,mc_logfc,sc_logfc,same_direction
8,IGKC,-1.895,-1.333,True
0,PTGDS,-1.624,-1.925,True
49,IGHG1,-1.452,-0.974,True
13,IGFBP7,-1.440,-1.222,True
14,MGP,-1.246,-1.221,True
19,DCN,-1.155,-1.196,True
24,LUM,-1.012,-1.135,True
12,COL3A1,-0.935,-1.242,True
60,BGN,-0.898,-0.900,True
47,COL1A2,-0.892,-0.993,True


## Sanity check: is collapse an artifact of the top-20 truncation?

`soft_assignments.npz` only keeps each cell's top-20-of-200 prototype scores
(`_topk_sparsify_rows`, done purely to bound file size on disk) before it's
saved and scored above. That function's own docstring notes a soft epsilon
can leave most of the 200 columns non-negligible per row -- so top-20
truncation could in principle be zeroing out real (if small) long-tail mass
on the ~190 "empty" prototypes, understating how spread out the model's
actual learned assignment is.

This recomputes `effective_n_metacells`/`gini`/`top5_share` on the raw,
untruncated softmax (`F.softmax(model.prototypes(z_all) / epsilon, dim=1)`)
for both cached scProto runs (`ema`, `nk`) -- no retraining, no re-encoding
needed beyond a forward pass. Compare against `size_eval`'s truncated numbers
above: if untruncated is much higher, the top-20 cutoff was hiding real
structure; if it is similar, the collapse is in the model's actual
assignment, not an artifact of how it was saved to disk.

In [84]:
from interpretable_ssl.evaluation.niche_program_recovery import gini, effective_n_metacells

untrunc_rows = []
for name, t in trainers_usage_mode.items():
    model = t.model
    model.eval()
    with torch.no_grad():
        z_all = t.encode_adata(t.train_ds.adata, model, z_idx=1).to(t.device)
        soft_full = F.softmax(model.prototypes(z_all) / t.epsilon, dim=1).cpu().numpy()
    del z_all

    n_k_full = soft_full.sum(axis=0)  # (K,) untruncated aggregate mass per prototype
    eff_n_full = effective_n_metacells(n_k_full)
    gini_full = gini(n_k_full)
    top5_full = np.sort(n_k_full)[::-1][:5].sum() / n_k_full.sum()
    n_near_zero = int((n_k_full < 1.0).sum())  # protos with <1 effective cell of mass, even untruncated

    untrunc_rows.append({
        'run': f'{name} (untruncated soft)',
        'effective_n_metacells': eff_n_full,
        'top5_share_of_cells': top5_full,
        'gini': gini_full,
        'n_protos_lt_1cell': n_near_zero,
    })
    print(f'{name}: UNTRUNCATED soft -- effective_n={eff_n_full:.1f}  gini={gini_full:.3f}  '
          f'top5_share={top5_full:.1%}  n_protos_with_mass<1cell={n_near_zero}/{soft_full.shape[1]}')

untrunc_df = pd.DataFrame(untrunc_rows).set_index('run')
print()
print('Truncated (top-20, from size_eval above) vs. untruncated, side by side:')
display(pd.concat([size_eval[['effective_n_metacells', 'top5_share_of_cells', 'gini']], untrunc_df.drop(columns='n_protos_lt_1cell')]).round(3))


  0%|          | 0/58 [00:00<?, ?it/s]

proto_bk08_np200_ema15: UNTRUNCATED soft -- effective_n=9.1  gini=0.932  top5_share=63.1%  n_protos_with_mass<1cell=0/200


  0%|          | 0/58 [00:00<?, ?it/s]

proto_bk08_np200_nk15: UNTRUNCATED soft -- effective_n=9.1  gini=0.881  top5_share=66.0%  n_protos_with_mass<1cell=0/200

Truncated (top-20, from size_eval above) vs. untruncated, side by side:


,effective_n_metacells,top5_share_of_cells,gini
run,,,
proto_bk08_np200_ema15 (soft),9.122,0.631,0.933
SEACells_arbf (soft),156.450,0.060,0.299
SEACells_bk08 (soft),155.391,0.066,0.292
proto_bk08_np200_ema15 (untruncated soft),9.147,0.631,0.932
proto_bk08_np200_nk15 (untruncated soft),9.059,0.660,0.881


## Beats-BOTH-baselines search: does scProto recover a niche program neither SEACells variant can?

The story this tests: `SEACells_arbf` has no spatial information at all (PCA-space
adaptive-RBF graph), so it structurally cannot separate niches within a cell type.
`SEACells_bk08` gets the exact same spatial graph scProto uses, but as a pure
archetypal method it has no transcriptional-reconstruction objective to tell a real
spatial-niche edge apart from a same-cell-type-boundary edge that just happens to be
spatially close (graph noise from BANKSY's own construction) -- it has no mechanism to
down-weight that. scProto, by contrast, is grounded in `L_rec` (real gene expression
reconstruction) throughout, which is the thing that could in principle let it use the
same noisy graph more selectively.

The "advantage" table above only required beating `max(arbf, bk08)" -- which a method
can do by beating just the easier one. This instead requires scProto to beat **both**
individually, tightens the robustness filter to `n_pos_mc>=3` (not >=1 -- we already
know 2-metacell wins are fragile given the collapse), and ranks by the *weaker* of the
two margins so a pair can't sneak in on one lucky comparison.

In [85]:
MARGIN_THRESH = 0.03

reliable_dual = per_pair_eval_detail[
    (per_pair_eval_detail['min_pos_mc_size'] >= 5) &
    (per_pair_eval_detail['n_real_cells'] >= 100) &
    (per_pair_eval_detail['n_pos_mc'] >= 3)
].copy()

dual_pivot = reliable_dual.pivot_table(index=['cell_type', 'niche'], columns='run', values='pearson_r')
purity_pivot = reliable_dual.pivot_table(index=['cell_type', 'niche'], columns='run', values='mean_niche_purity')

scp_col = 'proto_bk08_np200_ema15 (soft)'
arbf_col = 'SEACells_arbf (soft)'
bk08_col = 'SEACells_bk08 (soft)'

needed = [c for c in (scp_col, arbf_col, bk08_col) if c not in dual_pivot.columns]
if needed:
    print(f'WARNING -- missing columns for dual-baseline comparison: {needed}')
    dual = pd.DataFrame()
    beats_both = pd.DataFrame()
else:
    dual = dual_pivot[[scp_col, arbf_col, bk08_col]].dropna()
    dual['margin_vs_arbf'] = dual[scp_col] - dual[arbf_col]
    dual['margin_vs_bk08'] = dual[scp_col] - dual[bk08_col]
    dual['min_margin'] = dual[['margin_vs_arbf', 'margin_vs_bk08']].min(axis=1)
    dual['scproto_purity'] = purity_pivot[scp_col]

    def _bucket(r):
        # Only 'beats BOTH' is actual evidence for scProto's advantage claim (spatial
        # signal helps AND scProto extracts it better than plain archetypal analysis
        # on the same graph). Beating arbf alone but losing to bk08 is a genuine loss
        # (SEACells already covered in the earlier advantage table). Beating bk08 alone
        # while LOSING to arbf is actively evidence AGAINST scProto for that pair --
        # the no-spatial-info baseline still wins, so scProto's use of the graph isn't
        # even clearing the bar of ignoring it -- do not read this bucket as support.
        beats_a = r['margin_vs_arbf'] > MARGIN_THRESH
        beats_b = r['margin_vs_bk08'] > MARGIN_THRESH
        if beats_a and beats_b:
            return 'beats BOTH (real support)'
        elif beats_a:
            return 'beats arbf, loses to bk08 (SEACells wins overall)'
        elif beats_b:
            return 'beats bk08, loses to arbf (NOT support -- no-spatial baseline still wins)'
        else:
            return 'loses to both'
    dual['bucket'] = dual.apply(_bucket, axis=1)

    print(f'All {len(dual)} reliable pairs (min_pos_mc_size>=5, n_real_cells>=100, n_pos_mc>=3), '
          f'by which baseline(s) scProto actually beats (margin > {MARGIN_THRESH}):')
    display(dual.sort_values('min_margin', ascending=False).round(3))

    print()
    print('Counts per bucket (ONLY "beats BOTH" supports the advantage claim -- the rest are context/caveats):')
    display(dual['bucket'].value_counts())

    beats_both = dual[dual['bucket'].str.startswith('beats BOTH')].sort_values('min_margin', ascending=False)
    print()
    print(f'{len(beats_both)}/{len(dual)} pairs beat BOTH baselines -- these are the only real candidates '
          f'for the "scProto recovers a niche program neither SEACells variant can" claim:')
    display(beats_both.round(3))

All 5 reliable pairs (min_pos_mc_size>=5, n_real_cells>=100, n_pos_mc>=3), by which baseline(s) scProto actually beats (margin > 0.03):


run                              proto_bk08_np200_ema15 (soft)  \
cell_type   niche                                                
Macrophages Macrophage islands                           0.764   
Fibroblasts Desmoplastic stroma                          0.879   
            Tumor surface                                0.906   
Tumor cells Tumor surface                                0.136   
Fibroblasts Vascular stroma                              0.441   

run                              SEACells_arbf (soft)  SEACells_bk08 (soft)  \
cell_type   niche                                                             
Macrophages Macrophage islands                  0.526                 0.654   
Fibroblasts Desmoplastic stroma                 0.799                 0.809   
            Tumor surface                       0.792                 0.922   
Tumor cells Tumor surface                       0.182                 0.269   
Fibroblasts Vascular stroma                     0.672                 0.695   

run                              margin_vs_arbf  margin_vs_bk08  min_margin  \
cell_type   niche                                                             
Macrophages Macrophage islands            0.238           0.109       0.109   
Fibroblasts Desmoplastic stroma           0.080           0.070       0.070   
            Tumor surface                 0.114          -0.016      -0.016   
Tumor cells Tumor surface                -0.046          -0.132      -0.132   
Fibroblasts Vascular stroma              -0.231          -0.254      -0.254   

run                              scproto_purity  \
cell_type   niche                                 
Macrophages Macrophage islands            0.382   
Fibroblasts Desmoplastic stroma           0.496   
            Tumor surface                 0.445   
Tumor cells Tumor surface                 0.454   
Fibroblasts Vascular stroma               0.269   

run                                                                         bucket  
cell_type   niche                                                                   
Macrophages Macrophage islands                           beats BOTH (real support)  
Fibroblasts Desmoplastic stroma                          beats BOTH (real support)  
            Tumor surface        beats arbf, loses to bk08 (SEACells wins overall)  
Tumor cells Tumor surface                                            loses to both  
Fibroblasts Vascular stroma                                          loses to both


Counts per bucket (ONLY "beats BOTH" supports the advantage claim -- the rest are context/caveats):


bucket
beats BOTH (real support)                            2
loses to both                                        2
beats arbf, loses to bk08 (SEACells wins overall)    1
Name: count, dtype: int64


2/5 pairs beat BOTH baselines -- these are the only real candidates for the "scProto recovers a niche program neither SEACells variant can" claim:


,run,proto_bk08_np200_ema15 (soft),SEACells_arbf (soft),SEACells_bk08 (soft),margin_vs_arbf,margin_vs_bk08,min_margin,scproto_purity,bucket
cell_type,niche,,,,,,,,
Macrophages,Macrophage islands,0.764,0.526,0.654,0.238,0.109,0.109,0.382,beats BOTH (real support)
Fibroblasts,Desmoplastic stroma,0.879,0.799,0.809,0.080,0.070,0.070,0.496,beats BOTH (real support)


## Genes behind the beats-both candidates -- literature cross-check

Same `top_genes_for_pair` machinery as the candidates section above, restricted to
whichever pairs survive the beats-both filter -- these are the ones worth actually
checking against NSCLC tumor-microenvironment literature gene-by-gene.

In [86]:
N_TOP_DUAL = 5

for (ct, niche), row in beats_both.head(N_TOP_DUAL).iterrows():
    S, cell_ids = load_soft_assignments(run_dirs_eval['proto_bk08_np200_ema15'])
    mc_labels = soft_label_metacells(S, cell_ids, adata, CT_KEY, NICHE_KEY)
    pseudobulk = build_soft_pseudobulk(S, cell_ids, adata)
    genes = top_genes_for_pair(pseudobulk, mc_labels, ground_truth, ct, niche, top_n=15)
    print()
    print(f'=== {ct} x {niche}  (scProto r={row[scp_col]:.3f}, arbf r={row[arbf_col]:.3f}, '
          f'bk08 r={row[bk08_col]:.3f}, min_margin={row["min_margin"]:.3f}) ===')
    display(genes.round(3))


=== Macrophages x Macrophage islands  (scProto r=0.764, arbf r=0.526, bk08 r=0.654, min_margin=0.109) ===


,gene,mc_logfc,sc_logfc,same_direction
0,SPP1,3.102,2.155,True
87,IGKC,-2.268,-0.539,True
2,MARCO,2.020,1.694,True
18,PTGDS,-1.894,-1.030,True
48,INHBA,1.793,0.769,True
13,SELENOP,-1.597,-1.177,True
36,IL17RB,1.574,0.851,True
70,IL3RA,1.354,0.614,True
28,OLR1,1.348,0.928,True
54,CCND1,-1.289,-0.717,True



=== Fibroblasts x Desmoplastic stroma  (scProto r=0.879, arbf r=0.799, bk08 r=0.809, min_margin=0.070) ===


,gene,mc_logfc,sc_logfc,same_direction
4,IGHG2,1.518,1.112,True
5,IGHG1,1.402,1.100,True
20,CD55,-1.374,-0.641,True
11,COL15A1,-1.333,-0.848,True
0,IGFBP5,-1.294,-1.465,True
1,IGF1,1.240,1.324,True
3,IGKC,1.240,1.190,True
17,WNT2,1.037,0.772,True
15,COL14A1,0.992,0.788,True
2,PTGDS,0.972,1.239,True


## Genes behind the remaining scProto-unique pairs -- literature cross-check

Same `top_genes_for_pair` machinery, for the 3 scProto-unique pairs from the
coverage-union section above that haven't been checked against the NSCLC
source paper yet (Macrophages x Alveolar spaces and both Vascular endothelium
pairs were already pulled earlier in this notebook run).

In [ ]:
S, cell_ids = load_soft_assignments(run_dirs_eval['proto_bk08_np200_ema15'])
mc_labels = soft_label_metacells(S, cell_ids, adata, CT_KEY, NICHE_KEY)
pseudobulk = build_soft_pseudobulk(S, cell_ids, adata)

for ct, niche in [('Macrophages', 'Airways'), ('Macrophages', 'Desmoplastic stroma'),
                   ('Tumor cells', 'Desmoplastic stroma')]:
    genes = top_genes_for_pair(pseudobulk, mc_labels, ground_truth, ct, niche, top_n=15)
    print(f'\n=== {ct} x {niche} ===')
    display(genes.round(3))